In [1]:
!pip install -q langchain langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
# 🔐 Paste your Groq key
API_KEY = "gsk_*****************************"

import os
os.environ["GROQ_API_KEY"] = API_KEY
print("✅ Ready.")

✅ Ready.


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # 👈 the one line to change if deprecated
    api_key=API_KEY,
    temperature=0,                     # 0 = predictable, best for agents
)

In [4]:
from pydantic import BaseModel, Field
from typing import List

# 📦 the exact shape of a plan
class Plan(BaseModel):
    steps: List[str] = Field(description="ordered list of sub-tasks to solve the query")

planner = llm.with_structured_output(Plan)

query = "Compare the populations of Ludhiana and Amritsar, then say which is bigger."
plan = planner.invoke(
    f"Break this request into 2-4 simple sub-tasks. Request: {query}"
)

print("🧑✈️ Plan:")
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")

🧑✈️ Plan:
  1. Find the population of Ludhiana
  2. Find the population of Amritsar
  3. Compare the populations of Ludhiana and Amritsar
  4. Determine which city has a larger population


In [5]:
def worker(subtask: str) -> str:
    """🔧 handle ONE sub-task in isolation."""
    result = llm.invoke(f"Complete this single task concisely: {subtask}")
    return result.content

# run each step
results = []
for step in plan.steps:
    answer = worker(step)
    results.append(f"- {step}\n  → {answer}")
    print(f"🔧 {step}\n   {answer}\n")

# 🧑✈️ planner synthesizes the final answer
combined = "\n".join(results)
final = llm.invoke(
    f"Using these sub-results, give a final answer to '{query}':\n{combined}"
)
print("✅ FINAL:\n", final.content)

🔧 Find the population of Ludhiana
   The population of Ludhiana, as per the 2011 census, is approximately 1,618,879. However, the estimated population as of 2021 is around 2,065,000 (or 2.065 million).

🔧 Find the population of Amritsar
   The population of Amritsar is approximately 1.5 million people.

🔧 Compare the populations of Ludhiana and Amritsar
   As of the 2011 census, Ludhiana has a population of approximately 1,618,879, while Amritsar has a population of around 1,132,383.

🔧 Determine which city has a larger population
   I'd be happy to help, but you didn't provide the names of the two cities. Please provide the city names, and I'll do my best to determine which one has a larger population.

✅ FINAL:
 To compare the populations of Ludhiana and Amritsar, we can look at the most recent estimates provided. Ludhiana has an estimated population of around 2.065 million as of 2021, while Amritsar has a population of approximately 1.5 million. 

Based on these numbers, Ludhiana ha

In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 📦 1. Define the shared state
class State(TypedDict):
    topic: str
    draft: str

# ⚫ 2. Define three nodes (each returns a state UPDATE)
def node_draft(state: State):
    text = llm.invoke(f"Write one plain sentence about {state['topic']}.").content
    print("⚫ node_draft →", text)
    return {"draft": text}

def node_detail(state: State):
    text = llm.invoke(f"Add one specific detail to this: {state['draft']}").content
    print("⚫ node_detail →", text)
    return {"draft": text}

def node_polish(state: State):
    text = llm.invoke(f"Make this sound friendly and clear: {state['draft']}").content
    print("⚫ node_polish →", text)
    return {"draft": text}

# 🔧 3. Wire the graph
builder = StateGraph(State)
builder.add_node("draft",  node_draft)
builder.add_node("detail", node_detail)
builder.add_node("polish", node_polish)

builder.add_edge(START,   "draft")   # 🚦 start here
builder.add_edge("draft",  "detail")
builder.add_edge("detail", "polish")
builder.add_edge("polish", END)      # 🚦 stop here

graph = builder.compile()

# ▶️ 4. Run it
result = graph.invoke({"topic": "the LangGraph library", "draft": ""})
print("\n✅ FINAL DRAFT:\n", result["draft"])

⚫ node_draft → The LangGraph library is an open-source tool for constructing, manipulating, and analyzing large-scale knowledge graphs.
⚫ node_detail → The LangGraph library is an open-source tool for constructing, manipulating, and analyzing large-scale knowledge graphs, utilizing a graph database to store and query complex relationships between entities.
⚫ node_polish → Here's a friendly and clear version:

"Meet LangGraph, a powerful open-source library that helps you build, explore, and understand complex networks of information. It uses a graph database to store and query relationships between entities, making it easier to uncover insights and connections in large-scale knowledge graphs."

✅ FINAL DRAFT:
 Here's a friendly and clear version:

"Meet LangGraph, a powerful open-source library that helps you build, explore, and understand complex networks of information. It uses a graph database to store and query relationships between entities, making it easier to uncover insights an

In [7]:
from typing import Literal

def quality_router(state) -> Literal["rewrite", "done"]:
    # 🔀 read state, decide the next node's name
    if state["approved"]:
        return "done"
    return "rewrite"

In [8]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 📦 state carries the draft, a verdict, and a loop guard
class State(TypedDict):
    question: str
    answer: str
    approved: bool
    tries: int

# ⚫ write (or rewrite) an answer
def write(state: State):
    prompt = f"Answer clearly in 2 sentences: {state['question']}"
    if state.get("answer"):
        prompt = f"Improve this answer, make it clearer: {state['answer']}"
    ans = llm.invoke(prompt).content
    return {"answer": ans, "tries": state.get("tries", 0) + 1}

# ⚫ review: judge the draft, set approved True/False
def review(state: State):
    verdict = llm.invoke(
        f"Reply only YES if this is clear and correct, else NO:\n{state['answer']}"
    ).content.strip().upper()
    approved = verdict.startswith("YES") or state["tries"] >= 3   # 🛡️ loop guard
    print(f"🔎 review: verdict={verdict}  approved={approved}  (try {state['tries']})")
    return {"approved": approved}

# 🔀 router: loop back or finish
def route(state: State) -> Literal["rewrite", "done"]:
    return "done" if state["approved"] else "rewrite"

builder = StateGraph(State)
builder.add_node("write",  write)
builder.add_node("review", review)

builder.add_edge(START, "write")
builder.add_edge("write", "review")

# 🔀 THE conditional edge: from review, choose the next node by name
builder.add_conditional_edges(
    "review",              # after this node runs...
    route,                 # ...call this router...
    {                      # ...and map its return to a real node:
        "rewrite": "write",   # ↺ loop back
        "done": END,          # ✅ finish
    },
)

graph = builder.compile()

out = graph.invoke({"question": "How to make friends?",
                    "answer": "", "approved": False, "tries": 0})
print("\n✅ APPROVED ANSWER:\n", out["answer"])

🔎 review: verdict=YES  approved=True  (try 1)

✅ APPROVED ANSWER:
 To make friends, you should be open and approachable, engaging in conversations and activities that allow you to meet new people and build connections with those who share similar interests. By being yourself, listening actively, and showing genuine interest in others, you can establish meaningful relationships and develop strong, lasting friendships over time.


In [9]:
from langgraph.checkpoint.memory import InMemorySaver

# 💾 attach a checkpointer at compile time
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# 🧵 a thread_id labels this conversation's saved state
config = {"configurable": {"thread_id": "user-42"}}

# run once — state is saved automatically after each node
graph.invoke({"question": "What is LangGraph?",
              "answer": "", "approved": False, "tries": 0}, config)

# 🔍 later (even after a restart) — read back exactly where it stopped
saved = graph.get_state(config)
print("💾 saved state:", saved.values)

🔎 review: verdict=YES  approved=True  (try 1)
💾 saved state: {'question': 'What is LangGraph?', 'answer': 'LangGraph is an artificial intelligence (AI) model developed by Meta, designed to process and generate human-like language, with capabilities including text understanding, generation, and conversation. LangGraph is a large language model that uses machine learning algorithms to learn patterns and relationships in language, allowing it to perform tasks such as answering questions, translating text, and engaging in natural-sounding conversations.', 'approved': True, 'tries': 1}


In [10]:
from IPython.display import Image, display


In [11]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.9 MB/s eta 0:00:00


In [12]:
%%writefile streamlit_app.py
import streamlit as st
import os
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# --- LangGraph Setup (adapted from your notebook) ---

# IMPORTANT: This will get the API key from environment variables
# For Streamlit deployment, set this as a secret.
# For local Colab execution, it will use the existing os.environ["GROQ_API_KEY"]
API_KEY = os.environ.get("GROQ_API_KEY")

if not API_KEY:
    st.error("GROQ_API_KEY not found. Please set it as an environment variable or in Streamlit secrets.")
    st.stop()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=API_KEY,
    temperature=0,
)

# 📦 state carries the question, generated code, a verdict, and a loop guard
class State(TypedDict):
    question: str
    answer: str # This will now hold the generated Python code
    approved: bool
    tries: int

# ⚫ write (or rewrite) the Python code
def write(state: State):
    # Instruct the LLM to generate Python code directly
    prompt_for_code = (
        f"Generate a Python code snippet that answers the following request: {state['question']}. "
        f"Provide only the Python code, without any additional explanations or markdown formatting around the code. "
        f"Ensure the code is runnable and addresses the request directly."
    )
    if state.get("answer"): # If there's an existing answer (code), ask to improve it.
        prompt_for_code = (
            f"Improve this Python code snippet based on the following request: {state['question']}. "
            f"Current code: ```python\n{state['answer']}\n```. "
            f"Provide only the improved Python code, without any additional explanations or markdown formatting around the code."
        )

    ans = llm.invoke(prompt_for_code).content
    st.session_state.intermediate_steps.append(f"Drafting code (try {state.get('tries', 0) + 1}):\n```python\n{ans}\n```")
    return {"answer": ans, "tries": state.get("tries", 0) + 1}

# ⚫ review: judge the generated code, set approved True/False
def review(state: State):
    # Adjust review prompt for code
    verdict = llm.invoke(
        f"Reply only YES if this Python code is clear, correct, and directly answers the request, else NO.\nRequest: {state['question']}\nCode:\n```python\n{state['answer']}\n```"
    ).content.strip().upper()
    approved = verdict.startswith("YES") or state["tries"] >= 3  # 🛡️ loop guard
    st.session_state.intermediate_steps.append(f"Review verdict for try {state['tries']}: {verdict} (Approved: {approved})")
    return {"approved": approved}

# 🔀 router: loop back or finish
def route(state: State) -> Literal["rewrite", "done"]:
    return "done" if state["approved"] else "rewrite"

# Build the LangGraph
builder = StateGraph(State)
builder.add_node("write", write)
builder.add_node("review", review)

builder.add_edge(START, "write")
builder.add_edge("write", "review")
builder.add_conditional_edges(
    "review",
    route,
    {"rewrite": "write", "done": END},
)

graph = builder.compile()

# --- Streamlit App ---
st.title("Code Generation with LangGraph")
st.write("Ask a question, and I'll generate Python code for you!")

# Initialize session state for intermediate steps
if 'intermediate_steps' not in st.session_state:
    st.session_state.intermediate_steps = []

question = st.text_input("Your Question:", "")

if st.button("Generate Code"):
    if question:
        st.session_state.intermediate_steps = [] # Clear previous steps
        with st.spinner("Generating code..."):
            # Invoke the graph with the user's question
            out = graph.invoke({"question": question, "answer": "", "approved": False, "tries": 0})

            st.subheader("Generated Python Code:")
            st.code(out["answer"], language="python")

            if st.session_state.intermediate_steps:
                st.subheader("Generation Process:")
                for step_text in st.session_state.intermediate_steps:
                    st.info(step_text)
    else:
        st.warning("Please enter a question to generate code.")


Writing streamlit_app.py


In [13]:
# Run Streamlit in the background (output redirected to avoid conflicts)
!streamlit run streamlit_app.py &>/dev/null&